In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.models.resnet import BasicBlock
import torch.ao.quantization as quant
import types
import torch.ao.quantization as aq
from torchvision.models.resnet import BasicBlock
from torch.ao.quantization import get_default_qat_qconfig
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
# from torch.ao.quantization.pt2e import prepare_qat_pt2e, convert_pt2e
# from torch.ao.quantization.quantizer.qat_config import QATConfig
from torch.ao.quantization.quantizer.xnnpack_quantizer import XNNPACKQuantizer
import brevitas.nn as qnn
import copy
from torch.quantization.fake_quantize import FakeQuantizeBase

import os
import logging
from datetime import datetime

In [4]:
os.makedirs("./data", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./trained_models", exist_ok=True)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100,
                                         shuffle=False, num_workers=2)

## Full ResNet18 Traininig

In [ ]:
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

In [7]:
def training_loop(model, model_name, trainloader, testloader, num_epochs = 10):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}_{start_of_training_timestamp}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        # Log metrics
        logging.info(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )
    return start_of_training_timestamp

In [ ]:
start_of_training_timestamp = training_loop(model, "resnet18_cifar", trainloader, testloader)

In [ ]:
model_path = f"./trained_models/resnet18_cifar10_{start_of_training_timestamp}.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved as {model_path}")

size_bytes = os.path.getsize(model_path)
size_mb = size_bytes / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

## QAT ResNet Training

In [ ]:
qat_model = resnet18()
qat_model.fc = nn.Linear(qat_model.fc.in_features, 10)

if isinstance(trainset, torchvision.datasets.CIFAR10):
    qat_model.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        stride=1,          
        padding=1,          
        bias=False
    )

    # Remove maxpool (not needed for small inputs)
    qat_model.maxpool = nn.Identity()

In [ ]:
print(qat_model)

#### [DEPRECATED] QAT via torch.ao.quantization.prepare_qat (Eager way, older)

In [ ]:
class QuantizableBasicBlock(nn.Module):
    def __init__(self, orig_block: BasicBlock):
        super().__init__()
        # reuse original parameters / submodules
        self.conv1 = orig_block.conv1
        self.bn1 = orig_block.bn1
        self.relu = orig_block.relu
        self.conv2 = orig_block.conv2
        self.bn2 = orig_block.bn2
        self.downsample = orig_block.downsample  # may be None
        self.stride = orig_block.stride

        # local quant/dequant stubs (these use observers when prepared)
        self.quant = aq.QuantStub()
        self.dequant = aq.DeQuantStub()

    def forward(self, x):
        identity = x

        # Quantize at block entry => convs will be fake-quantized during QAT
        out = self.quant(x)

        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Dequantize so addition runs in FP32
        out = self.dequant(out)

        if self.downsample is not None:
            # keep downsample in FP32 by applying it directly on FP32 input
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

In [ ]:
def make_blocks_quantizable(model):
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for i in range(len(layer)):
            orig_block = layer[i]
            layer[i] = QuantizableBasicBlock(orig_block)

make_blocks_quantizable(qat_model)

In [ ]:
default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = default_qconfig

qat_model.conv1.qconfig = None
qat_model.fc.qconfig = None

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(qat_model, layer_name)
    for block in layer:
        if getattr(block, 'downsample', None) is not None:
            # set downsample (Sequential) to FP32
            block.downsample.qconfig = None

In [ ]:
qat_model_prepared = torch.ao.quantization.prepare_qat(qat_model)
print(qat_model_prepared)

#### [DEPRECATED] QAT via prepare_qat_fx (New (FX) way. lets you fine-tune per-module quantization in a declarative way)

In [ ]:
from torch.ao.quantization.qconfig import QConfig

class TernaryFakeQuantize(FakeQuantizeBase):
    def __init__(self, threshold=0.05):
        super().__init__()
        self.threshold = threshold  # values near zero → quantize to 0

    def forward(self, X):
        # Quantize weights/activations to {-1, 0, 1}
        X = torch.tanh(X)  # optional normalization step
        out = torch.zeros_like(X)
        out[X > self.threshold] = 1.0
        out[X < -self.threshold] = -1.0
        return out

    def _load_from_state_dict(self, *args, **kwargs):
        # Required to be compatible with torch FX QAT API
        pass
ternary_qconfig = QConfig(
    activation=TernaryFakeQuantize.with_args(threshold=0.05),
    weight=TernaryFakeQuantize.with_args(threshold=0.05)
)

default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = ternary_qconfig # default_qconfig
example_input = torch.randn(1, 3, 32, 32) # For checking 


qconfig_dict = {
    "": ternary_qconfig, # default_qconfig,  # default for all layers
    "module_name": [("conv1", None), ("fc", None)]  # disable quant for first and last
}

qat_model_prepared = prepare_qat_fx(qat_model, qconfig_dict, example_input)
print(qat_model_prepared)

#### QAT via torch.ao.quantization.pt2e (Recommened)

In [ ]:
# https://github.com/Xilinx/brevitas

example_input = (torch.randn(1, 3, 32, 32),)
exported = torch.export.export(qat_model, example_input).module()

quantizer = XNNPACKQuantizer()

In [ ]:
ternary_qat = QATConfig(
    activation_dtype=torch.quint8,  # activations stay 8-bit
    weight_dtype=torch.qint2,       # 2-bit weights → ternary
    enable_observer=True,
    enable_fake_quant=True,
    observer_kwargs={"quant_min": -1, "quant_max": 1}  # enforce ternary range
)

# --- FP32 config for first & last ---
fp32_qat = QATConfig(
    activation_dtype=None,
    weight_dtype=None,
    enable_fake_quant=False,
    enable_observer=False
)

In [ ]:
quantizer.set_global(ternary_qat)

# Disable quantization for first conv and final fc
quantizer.set_module_name("conv1", fp32_qat)
quantizer.set_module_name("fc", fp32_qat)

In [ ]:
qat_prepared = prepare_qat_pt2e(exported, quantizer)

#### QAT Model traininig

In [ ]:
qat_model_prepared.to(device)
start_of_training_timestamp = training_loop(qat_model_prepared, "resnet18_cifar10_qat", trainloader, testloader, num_epochs=1)

In [ ]:
qat_model_prepared.eval()
qat_model_prepared.to("cpu")
# final_quantized_model = convert_fx(qat_model_prepared)
final_quantized_model = convert_pt2e(qat_prepared)

qt_path = f"./trained_models/resnet18_cifar10_qat_{start_of_training_timestamp}.pth"
torch.save(final_quantized_model.state_dict(), qt_path)

print(f"Quantized model saved as {qt_path}")
print("Quantized file size (MB):", os.path.getsize(qt_path)/(1024**2))

In [ ]:
print(final_quantized_model)

#### QAT Model evaluation

In [ ]:
final_quantized_model.load_state_dict(torch.load("./trained_models/resnet18_cifar10_qat_07.10.2025-15:48:04.pth", map_location="cpu"))
final_quantized_model.eval()

In [ ]:
test_loss = 0.0
correct_test = 0
total_test = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to('cpu'), labels.to('cpu')
        outputs = final_quantized_model(inputs)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

test_loss /= total_test
test_acc = 100. * correct_test / total_test

# Log metrics
print(
    f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
)

In [ ]:
print(final_quantized_model)

#### Ternary quantizations with bravitas

In [ ]:
def apply_brevitas_quantization(model):
    for name, module in model.named_children():
        # if name in ["conv1", "fc"]:
        #     continue
        if isinstance(module, nn.Conv2d):
            # Create a new QuantConv2d layer with ternary weight quantization
            quantized_conv = qnn.QuantConv2d(
                in_channels=module.in_channels,
                out_channels=module.out_channels,
                kernel_size=module.kernel_size,
                stride=module.stride,
                padding=module.padding,
                dilation=module.dilation,
                groups=module.groups,
                bias=(module.bias is not None),
                # This is the key part for ternary quantization!
                weight_quant = ,  
            )
            # Copy the weights and bias from the original layer
            # Brevitas handles the quantization of these full-precision weights
            quantized_conv.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                quantized_conv.bias.data.copy_(module.bias.data)
            
            # Replace the original layer
            setattr(model, name, quantized_conv)
            
        elif len(list(module.children())) > 0:
            # Recurse for submodules (like BasicBlocks in ResNet)
            apply_brevitas_quantization(module)

In [ ]:
qat_ternary_model = copy.deepcopy(qat_model)
apply_brevitas_quantization(qat_ternary_model)

print(qat_ternary_model)

In [19]:
from torch import nn
from torch.nn import Module
import torch.nn.functional as F

import brevitas.nn as qnn


class QuantWeightLeNet(Module):
    def __init__(self):
        super(QuantWeightLeNet, self).__init__()
        self.conv1 = qnn.QuantConv2d(3, 6, 5, bias=True, weight_bit_width=2)
        self.relu1 = nn.ReLU()
        self.conv2 = qnn.QuantConv2d(6, 16, 5, bias=True, weight_bit_width=2)
        self.relu2 = nn.ReLU()
        self.fc1   = qnn.QuantLinear(16*5*5, 120, bias=True, weight_bit_width=2)
        self.relu3 = nn.ReLU()
        self.fc2   = qnn.QuantLinear(120, 84, bias=True, weight_bit_width=2)
        self.relu4 = nn.ReLU()
        self.fc3   = qnn.QuantLinear(84, 10, bias=True, weight_bit_width=2)

    def forward(self, x):
        out = self.relu1(self.conv1(x))
        out = F.max_pool2d(out, 2)
        out = self.relu2(self.conv2(out))
        out = F.max_pool2d(out, 2)
        out = out.reshape(out.shape[0], -1)
        out = self.relu3(self.fc1(out))
        out = self.relu4(self.fc2(out))
        out = self.fc3(out)
        return out

quant_weight_lenet = QuantWeightLeNet()
quant_weight_lenet.to("cuda")

QuantWeightLeNet(
  (conv1): QuantConv2d(
    3, 6, kernel_size=(5, 5), stride=(1, 1)
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
          (input_view_impl): Identity()
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameter(
              (view_shape_impl): OverTensorView()
            )
            (stats): _Stats(
              (stats_impl): AbsMax()
            )
          )
         

In [20]:
def training_loop_brevitas(model, model_name, trainloader, testloader, num_epochs = 10):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}_{start_of_training_timestamp}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        # Log metrics
        logging.info(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )
    return start_of_training_timestamp

training_loop(quant_weight_lenet, "lenet5_cifar10", trainloader, testloader, 10)

2025-10-14 09:58:01,368 [INFO] Epoch [1/10] Train Loss: 1.8431, Train Acc: 32.48% Test Loss: 1.6473, Test Acc: 40.42%
2025-10-14 09:58:15,211 [INFO] Epoch [2/10] Train Loss: 1.6573, Train Acc: 39.82% Test Loss: 1.6287, Test Acc: 43.17%
2025-10-14 09:58:29,548 [INFO] Epoch [3/10] Train Loss: 1.5646, Train Acc: 43.14% Test Loss: 1.4325, Test Acc: 49.27%
2025-10-14 09:58:44,663 [INFO] Epoch [4/10] Train Loss: 1.5052, Train Acc: 45.59% Test Loss: 1.4365, Test Acc: 48.06%
2025-10-14 09:58:59,941 [INFO] Epoch [5/10] Train Loss: 1.4677, Train Acc: 47.12% Test Loss: 1.3501, Test Acc: 51.67%
2025-10-14 09:59:15,108 [INFO] Epoch [6/10] Train Loss: 1.4441, Train Acc: 48.12% Test Loss: 1.3830, Test Acc: 51.20%
2025-10-14 09:59:29,925 [INFO] Epoch [7/10] Train Loss: 1.4201, Train Acc: 49.36% Test Loss: 1.2756, Test Acc: 54.75%
2025-10-14 09:59:43,800 [INFO] Epoch [8/10] Train Loss: 1.3779, Train Acc: 50.83% Test Loss: 1.3077, Test Acc: 53.44%
2025-10-14 09:59:57,024 [INFO] Epoch [9/10] Train Loss: 

'14.10.2025-09:57:46'

In [ ]:
from brevitas.export import export_onnx_qcdq
import torch

# Weight-only model
export_onnx_qcdq(quant_weight_lenet, torch.randn(1, 3, 32, 32), export_path='4b_weight_lenet.onnx')

# Weight-activation model
export_onnx_qcdq(quant_weight_act_lenet, torch.randn(1, 3, 32, 32), export_path='4b_weight_act_lenet.onnx')

# Weight-activation-bias model
export_onnx_qcdq(quant_weight_act_bias_lenet, torch.randn(1, 3, 32, 32), export_path='4b_weight_act_bias_lenet.onnx')